# Lab: Price a real AI task
**Solution notebook.** Reference solution. Try the starter notebook first.

## What you will do
You will estimate how many tokens a workplace task uses, check whether a long document fits in a model's context window, and price the task for a month.

**Time:** 20 to 25 minutes. **Needs:** Google Colab or Python 3.12+. No API key. The optional last step installs `tiktoken` 0.14.0 and needs internet access, which Colab has.

**Rule of thumb used here:** Google's Gemini documentation says one token is about 4 characters of English text, and 100 tokens is about 60 to 80 English words. Exact counts differ by model and by language.

In [ ]:
# Step 1. Inputs.
# Paid Standard prices for Gemini 3.5 Flash-Lite, from https://ai.google.dev/gemini-api/docs/pricing
#. Prices change: check the page and update these before relying on the result.
PRICE_INPUT_PER_1M = 0.30    # US dollars per 1 million input tokens
PRICE_OUTPUT_PER_1M = 2.50   # US dollars per 1 million output tokens (includes thinking tokens)
CHARS_PER_TOKEN = 4          # rule of thumb for English text

SAMPLE_TICKET = """Subject: Can't log in after password reset
Hi team, I reset my password this morning using the link in your email, but the
app still says my credentials are wrong. I'm on the latest version on my phone.
I tried twice and now I'm locked out. I need to approve payroll by 3 p.m. today.
Can someone help urgently? Thanks, Priya (Accounts Payable)"""

SAMPLE_SUMMARY = """Login fails after password reset; user now locked out.
Urgent: payroll approval due 3 p.m. today. Suggested action: unlock account,
confirm reset completed, escalate to priority queue."""
print(len(SAMPLE_TICKET), "characters in the ticket")

In [ ]:
# Step 2. Estimate tokens from characters.
import math

def estimate_tokens(text: str) -> int:
    """Estimate tokens as characters divided by CHARS_PER_TOKEN, rounded up."""
    return math.ceil(len(text) / CHARS_PER_TOKEN)

INSTRUCTIONS = "Summarize this support ticket in 3 short lines. Flag anything urgent."
input_tokens = estimate_tokens(INSTRUCTIONS) + estimate_tokens(SAMPLE_TICKET)
output_tokens = estimate_tokens(SAMPLE_SUMMARY)
print(f"Input tokens per ticket:  {input_tokens}")
print(f"Output tokens per ticket: {output_tokens}")

In [ ]:
# Step 3. Price the task for a month.
TICKETS_PER_DAY = 500
DAYS_PER_MONTH = 30

def monthly_cost(in_tok: int, out_tok: int, per_day: int, days: int) -> float:
    """Return the monthly cost in US dollars, rounded to 2 decimals."""
    calls = per_day * days
    cost = (in_tok * calls / 1_000_000) * PRICE_INPUT_PER_1M + (out_tok * calls / 1_000_000) * PRICE_OUTPUT_PER_1M
    return round(cost, 2)

cost = monthly_cost(input_tokens, output_tokens, TICKETS_PER_DAY, DAYS_PER_MONTH)
in_cost = input_tokens * PRICE_INPUT_PER_1M
out_cost = output_tokens * PRICE_OUTPUT_PER_1M
print(f"{TICKETS_PER_DAY * DAYS_PER_MONTH:,} tickets a month cost about ${cost:,.2f}")
print(f"Output tokens are {round(100 * output_tokens / (input_tokens + output_tokens))}% of tokens "
      f"but {round(100 * out_cost / (in_cost + out_cost))}% of the cost")

In [ ]:
# Step 4. Does a long document fit in the context window?
# A context window is the most tokens a model can handle in one request.
WORDS_PER_PAGE = 500
CHARS_PER_WORD = 6   # about 5 letters plus a space

def fits(pages: int, window_tokens: int, reserve_for_answer: int = 8_000) -> bool:
    """True if the document plus room for the answer fits in the window."""
    doc_tokens = math.ceil(pages * WORDS_PER_PAGE * CHARS_PER_WORD / CHARS_PER_TOKEN)
    return doc_tokens + reserve_for_answer <= window_tokens

for pages in (50, 300, 1500):
    for window in (128_000, 1_000_000):
        print(f"{pages:>5} pages in a {window:>9,}-token window: {'fits' if fits(pages, window) else 'does NOT fit'}")

In [ ]:
# Step 5 (optional, needs internet). Exact count with one real tokenizer.
# tiktoken is OpenAI's tokenizer library. Gemini uses a different tokenizer, so counts differ.
# For an exact Gemini count you call client.models.count_tokens, covered in Module 4.
RUN_OPTIONAL = False   # set to True in Colab
if RUN_OPTIONAL:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken==0.14.0"], check=True)
    import tiktoken
    enc = tiktoken.get_encoding("o200k_base")
    print(f"tiktoken count: {len(enc.encode(SAMPLE_TICKET))} | our estimate: {estimate_tokens(SAMPLE_TICKET)}")
else:
    print("Optional step skipped. Set RUN_OPTIONAL = True to compare with a real tokenizer.")

## Check your understanding
1. Output tokens were fewer than input tokens, yet most of the cost. Why?
2. A colleague wants to paste a 1,500-page contract archive into one prompt. What would you tell them?

## Apply it to your field
Pick one repeated task from your track (for example, summarizing lesson plans, claim notes or campaign briefs). Paste a realistic, non-confidential sample into `SAMPLE_TICKET`, set a realistic daily volume, and record the monthly cost in your project log. Note one way to cut the cost in half.

---
BJS Academy · Prompt to Agent (Course 03) · Dr. Bih Janet Shufor. Original course material. Learn AI. Shape tomorrow.